# AI Weather Forecast over Wyoming

Runs a 5-day deterministic forecast with NVIDIA **FourCastNet (FCN)** via [Earth2Studio](https://github.com/NVIDIA/earth2studio), initialized from **GFS** analysis, then crops and visualizes the result over Wyoming.

- Model: `FCN` (26-channel FourCastNet, 6h lead time step)
- Data source: `GFS` (near-real-time initial conditions)
- Region: Wyoming bounding box (lat 40.5–45.5°N, lon -111.5–-104°W)
- Variables saved/plotted: `t2m`, `u10m`, `v10m`, `msl`
- Requires a CUDA GPU.

## 1. Setup

Requires `earth2studio` installed with the `fcn` and `data` extras, e.g.:
```bash
uv pip install "earth2studio[fcn,data]"
```

In [ ]:
from collections import OrderedDict
from datetime import datetime, timedelta

import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from earth2studio.models.px import FCN
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.run import deterministic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration

Wyoming's bounding box (approximate state extent) and forecast settings.

In [ ]:
# Wyoming bounding box (degrees)
WYOMING_LAT_MIN, WYOMING_LAT_MAX = 40.5, 45.5
WYOMING_LON_MIN, WYOMING_LON_MAX = -111.5, -104.0  # GFS/FCN use 0-360 longitude internally

# Forecast configuration
FORECAST_HOURS = 5 * 24  # 5-day forecast
MODEL_STEP_HOURS = 6
NSTEPS = FORECAST_HOURS // MODEL_STEP_HOURS  # 20 steps

# Initial condition time (most recent GFS cycle available, UTC).
# Round down to the nearest 6-hour GFS cycle (00/06/12/18Z) a few hours in the past
# so the data is guaranteed to be published.
now = datetime.utcnow() - timedelta(hours=6)
init_time = now.replace(hour=(now.hour // 6) * 6, minute=0, second=0, microsecond=0)
INIT_TIME = init_time.strftime("%Y-%m-%dT%H:%M:%S")

OUTPUT_PATH = "wyoming_forecast.zarr"

print(f"Initial condition time: {INIT_TIME}")
print(f"Forecast length: {FORECAST_HOURS}h ({NSTEPS} steps @ {MODEL_STEP_HOURS}h)")

## 3. Load model and data source

In [ ]:
# FourCastNet prognostic model
package = FCN.load_default_package()
model = FCN.load_model(package)

# GFS analysis/forecast data source for initial conditions
data = GFS()

# Zarr output store
io = ZarrBackend(OUTPUT_PATH)

## 4. Run the forecast

Only `t2m`, `u10m`, `v10m`, and `msl` are kept in the output (the model still uses its full 26-variable state internally).

In [ ]:
output_coords = OrderedDict(
    {"variable": np.array(["t2m", "u10m", "v10m", "msl"])}
)

io = deterministic(
    time=[INIT_TIME],
    nsteps=NSTEPS,
    prognostic=model,
    data=data,
    io=io,
    output_coords=output_coords,
    device=device,
)

print("Forecast complete. Variables in store:", list(io.root.array_keys()))

## 5. Load output and crop to Wyoming

In [ ]:
ds = xr.open_zarr(OUTPUT_PATH)

# Earth2Studio stores longitude on a 0-360 grid; convert bbox to match
lon_min_360 = WYOMING_LON_MIN % 360
lon_max_360 = WYOMING_LON_MAX % 360

ds_wy = ds.sel(
    lat=slice(WYOMING_LAT_MAX, WYOMING_LAT_MIN),  # lat is typically descending
    lon=slice(lon_min_360, lon_max_360),
)

ds_wy

## 6. Visualize

2m temperature and 10m wind for the final forecast lead time, cropped to Wyoming.

In [ ]:
lead_idx = -1  # last forecast step
lead_hours = int(ds_wy.lead_time.values[lead_idx] / np.timedelta64(1, "h"))

t2m = ds_wy["t2m"].isel(time=0, lead_time=lead_idx) - 273.15  # K -> C
u10m = ds_wy["u10m"].isel(time=0, lead_time=lead_idx)
v10m = ds_wy["v10m"].isel(time=0, lead_time=lead_idx)

fig, ax = plt.subplots(
    figsize=(8, 7), subplot_kw={"projection": ccrs.PlateCarree()}
)
im = t2m.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="turbo",
    cbar_kwargs={"label": "2m Temperature (\u00b0C)"},
)
ax.quiver(
    u10m.lon, u10m.lat, u10m.values, v10m.values,
    transform=ccrs.PlateCarree(), regrid_shape=15, color="black",
)
ax.add_feature(cfeature.STATES, edgecolor="black", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, edgecolor="black")
ax.set_extent(
    [WYOMING_LON_MIN, WYOMING_LON_MAX, WYOMING_LAT_MIN, WYOMING_LAT_MAX],
    crs=ccrs.PlateCarree(),
)
ax.set_title(f"FCN forecast: t2m + 10m wind over Wyoming\nInit {INIT_TIME}Z, +{lead_hours}h")
plt.tight_layout()
plt.show()

In [ ]:
# Mean sea level pressure evolution over Wyoming (spatial mean per lead time)
msl_mean = ds_wy["msl"].isel(time=0).mean(dim=["lat", "lon"]) / 100.0  # Pa -> hPa
lead_hours_all = (ds_wy.lead_time.values / np.timedelta64(1, "h")).astype(int)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lead_hours_all, msl_mean.values, marker="o")
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("Mean MSL pressure over Wyoming (hPa)")
ax.set_title(f"FCN forecast: Wyoming-average MSL pressure\nInit {INIT_TIME}Z")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save the cropped Wyoming subset

Full global output is already saved to `wyoming_forecast.zarr` above (despite the name, it contains the global forecast). This saves just the cropped region to a lightweight NetCDF file for downstream use.

In [ ]:
ds_wy.to_netcdf("wyoming_forecast_subset.nc")
print("Saved cropped Wyoming forecast to wyoming_forecast_subset.nc")